# NB06 Merkle trees — Chapter 6

A chapter notebook for *Blockchain for Lawyers: A Reading Through Technology and Economics*. The code is the same as in the text.

- All code here is a **toy implementation** meant to make a principle visible. Do not use it with real assets, real private keys or real transactions.
- No external API is called, and everything runs without a network. Cells that need the bundled snapshot data carry that data inside the cell.
- Standard library only (run and checked on Python 3.13). There is no install cell.
- The outputs are the editor's own run, stored as produced. Running the cells again should give the same values (random seeds are fixed).

In [ ]:
# NB06-1: Merkle tree — computing the root, making an inclusion proof, verifying the proof (general form: SHA-256 once)
import hashlib, math

def H(b: bytes) -> bytes:
    return hashlib.sha256(b).digest()

def merkle_levels(leaves):
    """Return the hash list of each level, from the bottom level up, given the list of leaf hashes.
    If a level has an odd count, use the last one once more to make a pair (the same rule as Bitcoin; see Section 4)."""
    levels = [leaves]
    while len(levels[-1]) > 1:
        cur = levels[-1]
        if len(cur) % 2 == 1:
            cur = cur + [cur[-1]]
        levels.append([H(cur[i] + cur[i + 1]) for i in range(0, len(cur), 2)])
    return levels

def merkle_proof(levels, index):
    """Inclusion proof for leaf number index: a list of (sibling hash, is the sibling on the right) — the 'handedness' goes with it."""
    proof = []
    for level in levels[:-1]:
        if len(level) % 2 == 1:
            level = level + [level[-1]]
        sibling = index ^ 1                         # sibling's position: the right neighbor if index is even, the left neighbor if odd
        proof.append((level[sibling], sibling > index))
        index //= 2
    return proof

def verify_proof(leaf, proof, root):
    h = leaf
    for sibling, sibling_is_right in proof:
        h = H(h + sibling) if sibling_is_right else H(sibling + h)
    return h == root

txs = [f"tx{i}: Alice pays Bob {i} coin".encode() for i in range(8)]
leaves = [H(t) for t in txs]
levels = merkle_levels(leaves)
root = levels[-1][0]
print("8 transactions → hashes per level:", [len(l) for l in levels], "→ root", root.hex()[:16] + "…")

idx = 4                                            # the fifth transaction (4, counting from 0)
proof = merkle_proof(levels, idx)
print("length of the inclusion proof for tx4:", len(proof), "hashes =", 32 * len(proof), "bytes")
print("verify proof (original tx4):", verify_proof(leaves[idx], proof, root))
print("verify proof (tampered tx4):", verify_proof(H(b"tx4: Alice pays Bob 40 coin"), proof, root))

tampered = leaves[:]; tampered[2] = H(b"tx2: Alice pays Bob 200 coin")
print("does changing tx2 change the root:", merkle_levels(tampered)[-1][0] != root)

8 transactions → hashes per level: [8, 4, 2, 1] → root 23db046cc041f456…
length of the inclusion proof for tx4: 3 hashes = 96 bytes
verify proof (original tx4): True
verify proof (tampered tx4): False
does changing tx2 change the root: True


In [ ]:
# NB06-2: how much does a proof grow as the number of transactions grows — a single-line chain (N steps) versus a tree (log N)
import math
print(f"{'tx count n':>12} {'chain walk (max steps)':>22} {'Merkle proof hashes':>20} {'bytes':>10}")
for n in [1, 2, 8, 100, 1_000, 4_000, 10_000, 100_000, 1_000_000]:
    k = math.ceil(math.log2(n)) if n > 1 else 0
    print(f"{n:>12,} {n:>22,} {k:>20} {32*k:>10,}")

  tx count n chain walk (max steps)  Merkle proof hashes      bytes
           1                      1                    0          0
           2                      2                    1         32
           8                      8                    3         96
         100                    100                    7        224
       1,000                  1,000                   10        320
       4,000                  4,000                   12        384
      10,000                 10,000                   14        448
     100,000                100,000                   17        544
   1,000,000              1,000,000                   20        640


In [ ]:
# NB06-3: Bitcoin-style Merkle tree — double SHA-256, odd duplication, and the "same root" of CVE-2012-2459
import hashlib

def H2(b: bytes) -> bytes:                         # Bitcoin's hash: SHA256(SHA256(x))
    return hashlib.sha256(hashlib.sha256(b).digest()).digest()

def bitcoin_merkle_root(txids):
    """Compute the root from a list of txids (internal byte order). If the count is odd, duplicate the last one."""
    layer = list(txids)
    while len(layer) > 1:
        if len(layer) % 2 == 1:
            layer.append(layer[-1])
        layer = [H2(layer[i] + layer[i + 1]) for i in range(0, len(layer), 2)]
    return layer[0]

ids = [H2(bytes([i])) for i in range(1, 7)]        # fake txids for transactions 1-6
root_a = bitcoin_merkle_root(ids)
root_b = bitcoin_merkle_root(ids + ids[4:6])       # [1,2,3,4,5,6,5,6]
print("[1..6] root    :", root_a.hex())
print("[1..6,5,6] root:", root_b.hex())
print("same root for both lists:", root_a == root_b, " ← different transaction lists, same root")

# in a block with only one transaction (the genesis block is one), root = that transaction's txid
one = H2(b"only one transaction")
print("with 1 transaction, root == txid :", bitcoin_merkle_root([one]) == one, "(checked with real data in Chapter 12)")

[1..6] root    : df316ddf151a6d7f65a868df97de2168ed674371119b038bb97c9926b3f7948d
[1..6,5,6] root: df316ddf151a6d7f65a868df97de2168ed674371119b038bb97c9926b3f7948d
same root for both lists: True  ← different transaction lists, same root
with 1 transaction, root == txid : True (checked with real data in Chapter 12)
